# Compare three ResNet34 members on the shared 100 studies

Attach **three model variations**, **one completed balanced-holdout output dataset** (member 1 or 2), and **`gany24558/rsna-knee-normalized-training-data`**. Enable a GPU; Internet can remain off. No training or uploads occur.

The one output dataset supplies `split_plan_private.json`, `original_split_plan_private.json`, and `selection_targets_private.csv`. The notebook regenerates predictions from all three model packages on the same 100 selection studies, then compares individuals, pairs and the trio. Images are required as well as model weights. No separate member-0/member-1 prediction datasets are required.

Attach the intended **final model version** of each variation. Auto-discovery only accepts a single package per member; use explicit paths if multiple versions are attached. The model manifest identifies the member, split plan, dataset and best epoch, but does not prove early stopping: check the printed epochs against your completed runs (0: 15, 1: 10, 2: 12).

BCE measures agreement with community soft targets, not verified clinical accuracy. This is a selection set, not an independent test. Small differences may be noise. Compare equal-weight blends before tuning weights; inference runtime on the competition test set still needs separate benchmarking. Keep all outputs private.


In [1]:
from pathlib import Path
# Pin actual mounted package directories here if discovery is ambiguous.
MODEL_PACKAGES = {0: None, 1: None, 2: None}
# Directory containing the completed member-1 or member-2 split plan and selection targets.
SAVED_OUTPUT = None
DATASET_ROOT = None
EXPECTED_BEST_EPOCHS = {0: 15, 1: 10, 2: 12}
OUTPUT = Path('/kaggle/working/ensemble_comparison')
MAX_HOURS = 2.0


In [2]:
from pathlib import Path
from itertools import combinations
import json
import numpy as np
import pandas as pd


def compare_selection_members(member_dirs, output_dir):
    """Compare equal-weight probability blends against identical masked soft targets."""
    if set(member_dirs) != {0, 1, 2}:
        raise ValueError('Provide completed output directories for members 0, 1 and 2.')
    id_col = 'StudyInstanceUID'
    reference = None
    predictions = {}
    provenance = {}
    for member, directory in sorted(member_dirs.items()):
        directory = Path(directory)
        def read_json(name):
            return json.loads((directory / name).read_text())
        identity = read_json('run_identity.json')
        status = read_json('training_status.json')
        plan = read_json('split_plan_private.json')
        if identity['member'] != member:
            raise ValueError(f'Member {member}: directory belongs to another member.')
        if status['reason'] not in ('early_stopping', 'max_epochs', 'regenerated_best_checkpoint'):
            raise ValueError(f'Member {member}: training is incomplete.')
        ids = [str(x) for x in plan['selection_ids']]
        if len(ids) != 100 or len(set(ids)) != 100:
            raise ValueError(f'Member {member}: expected 100 unique selection studies.')
        for split in plan['members']:
            if set(ids) & set(map(str, split['train_ids'] + split['validation_ids'])):
                raise ValueError(f'Member {member}: selection leakage in saved plan.')
        def read_table(name):
            frame = pd.read_csv(directory / name, dtype={id_col: str})
            if frame[id_col].isna().any() or not frame[id_col].is_unique:
                raise ValueError(f'Member {member}: missing or duplicate study IDs in {name}.')
            if set(frame[id_col]) != set(ids):
                raise ValueError(f'Member {member}: selection IDs differ in {name}.')
            return frame.set_index(id_col).sort_index()
        pred = read_table('selection_predictions_private.csv')
        targets = read_table('selection_targets_private.csv')
        columns = list(pred.columns)
        if len(columns) != 12 or len(set(columns)) != 12:
            raise ValueError(f'Member {member}: expected 12 prediction columns.')
        if reference is not None:
            if set(columns) != set(reference[1]):
                raise ValueError('Members have different finding columns.')
            columns = reference[1]
        masks = [t + '__mask' for t in columns]
        raw_mask = targets[masks]
        if raw_mask.isna().any().any() or not raw_mask.isin([True, False, 0, 1]).all().all():
            raise ValueError(f'Member {member}: invalid label masks.')
        mask = raw_mask.to_numpy(bool)
        y = targets[columns].to_numpy(float)
        p = pred[columns].to_numpy(float)
        if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
            raise ValueError(f'Member {member}: invalid probabilities.')
        if not np.isfinite(y[mask]).all() or ((y[mask] < 0) | (y[mask] > 1)).any():
            raise ValueError(f'Member {member}: invalid active soft targets.')
        if not mask.any() or not mask.any(axis=0).all():
            raise ValueError('Every finding needs at least one active target.')
        if reference is None:
            reference = (list(pred.index), columns, y, mask)
        else:
            if list(pred.index) != reference[0] or not np.array_equal(mask, reference[3]):
                raise ValueError('Members have different study IDs or label masks.')
            if not np.allclose(y[mask], reference[2][mask], rtol=0, atol=1e-8):
                raise ValueError('Members have different selection targets.')
        saved = read_json('selection_metrics_private.json')
        clipped = np.clip(p, 1e-7, 1-1e-7)
        loss = -(y*np.log(clipped) + (1-y)*np.log(1-clipped))
        if not np.isclose(loss[mask].mean(), saved['soft_target_bce'], rtol=0, atol=1e-6):
            raise ValueError(f'Member {member}: saved selection BCE does not reproduce.')
        predictions[member] = p
        provenance[str(member)] = dict(directory=str(directory), best_epoch=status['best_epoch'])
    ids, columns, y, mask = reference
    rows, per_label = [], []
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for size in (1, 2, 3):
        for members in combinations(sorted(predictions), size):
            name = '+'.join(map(str, members))
            p = np.mean([predictions[m] for m in members], axis=0)
            q = np.clip(p, 1e-7, 1-1e-7)
            losses = -(y*np.log(q) + (1-y)*np.log(1-q))
            rows.append(dict(members=name, model_count=size, studies=len(ids),
                             soft_target_bce=float(losses[mask].mean())))
            for j, finding in enumerate(columns):
                per_label.append(dict(members=name, finding=finding,
                    soft_target_bce=float(losses[mask[:, j], j].mean())))
            pd.DataFrame(p, index=pd.Index(ids, name=id_col), columns=columns).to_csv(
                output_dir / f'members_{name.replace("+", "_")}_predictions_private.csv')
    results = pd.DataFrame(rows).sort_values('soft_target_bce').reset_index(drop=True)
    best_single = results.loc[results.model_count == 1, 'soft_target_bce'].min()
    results['bce_delta_vs_best_single'] = results.soft_target_bce - best_single
    results.to_csv(output_dir / 'ensemble_comparison_private.csv', index=False)
    pd.DataFrame(per_label).to_csv(output_dir / 'ensemble_per_label_private.csv', index=False)
    (output_dir / 'comparison_inputs_private.json').write_text(json.dumps(provenance, indent=2))
    return results


In [3]:
import sys, importlib, hashlib, time
import torch

def unique(paths, description):
    paths = sorted(set(paths))
    if len(paths) != 1:
        raise ValueError(f'{description}: expected one path, found {paths}. Set its explicit path.')
    return paths[0]

root = Path('/kaggle/input')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU to regenerate predictions from model weights.')
if SAVED_OUTPUT is None:
    SAVED_OUTPUT = unique([p.parent for p in root.rglob('selection_targets_private.csv')
        if (p.parent/'original_split_plan_private.json').is_file()], 'SAVED_OUTPUT')
SAVED_OUTPUT = Path(SAVED_OUTPUT)
plan = json.loads((SAVED_OUTPUT/'split_plan_private.json').read_text())
original = json.loads((SAVED_OUTPUT/'original_split_plan_private.json').read_text())
if set(plan['selection_ids']) != set(original['selection_ids']):
    raise ValueError('Original and revised selection IDs differ.')
target_file = SAVED_OUTPUT/'selection_targets_private.csv'
ids = list(map(str, plan['selection_ids']))
if len(ids) != 100 or len(set(ids)) != 100:
    raise ValueError('Expected exactly 100 distinct selection IDs.')
packages = {}
manifests = {}
for member, explicit in MODEL_PACKAGES.items():
    if explicit is None:
        candidates = []
        for file in (root/'models').rglob('manifest.json'):
            manifest = json.loads(file.read_text())
            if (manifest.get('model_family') == 'resnet34-study-mean-max-v1'
                and manifest.get('training_identity', {}).get('member') == member):
                candidates.append(file.parent)
        explicit = unique(candidates, f'MODEL_PACKAGES[{member}]')
    package = Path(explicit)
    manifest = json.loads((package/'manifest.json').read_text())
    if manifest['training_identity']['member'] != member:
        raise ValueError(f'Wrong member in {package}')
    if manifest['best_epoch'] != EXPECTED_BEST_EPOCHS[member]:
        raise ValueError(f'Member {member}: unexpected best epoch; attach the final model version.')
    for name, digest in manifest['files'].items():
        file = (package/name).resolve()
        if not file.is_relative_to(package.resolve()):
            raise ValueError('Unsafe manifest path')
        if hashlib.sha256(file.read_bytes()).hexdigest() != digest:
            raise ValueError(f'Package checksum mismatch: {file}')
    packages[member], manifests[member] = package, manifest
    print('Member', member, 'best epoch', manifest['best_epoch'], 'package', package)
# Use identical exported runtime across members; refuse silent implementation drift.
for name in ('cnn_runtime.py', 'knee_data.py'):
    if len({(p/name).read_bytes() for p in packages.values()}) != 1:
        raise ValueError(f'Member runtimes differ: {name}; audit before comparing.')
sys.path.insert(0, str(packages[0]))
for name in ('cnn_runtime', 'knee_data'):
    sys.modules.pop(name, None)
data = importlib.import_module('knee_data')
cnn = importlib.import_module('cnn_runtime')
bundle = data.discover_bundle(DATASET_ROOT, dataset_handle='gany24558/rsna-knee-normalized-training-data')
series, _, _, index = data.load_bundle(bundle)
if not set(ids).issubset(set(series[data.ID])):
    raise ValueError('Selection studies are missing from normalized images.')
by_study = {uid: g.sort_values(data.SID).to_dict('records') for uid, g in series.groupby(data.ID) if uid in set(ids)}
targets = pd.read_csv(target_file, dtype={data.ID: str}).set_index(data.ID)
if not targets.index.is_unique or set(targets.index) != set(ids):
    raise ValueError('Selection targets do not match the saved plan.')
targets = targets.loc[ids]
y = targets[data.TARGETS].to_numpy(float)
raw_mask = targets[[t+'__mask' for t in data.TARGETS]]
if raw_mask.isna().any().any() or not raw_mask.isin([True,False,0,1]).all().all():
    raise ValueError('Invalid target masks')
mask = raw_mask.to_numpy(bool)
if not mask.any() or not np.isfinite(y[mask]).all() or ((y[mask]<0)|(y[mask]>1)).any():
    raise ValueError('Invalid active targets')
deadline = time.monotonic() + MAX_HOURS*3600
member_dirs = {}
for member, package in packages.items():
    manifest = manifests[member]
    member_plan = original if member == 0 else plan
    identity = manifest['training_identity']
    if identity['repeated_holdout_plan_hash'] != data.fingerprint(member_plan):
        raise ValueError(f'Member {member}: split plan does not match model.')
    if identity['dataset'] != index['identity'] or manifest['targets'] != data.TARGETS:
        raise ValueError(f'Member {member}: normalized dataset or target order mismatch.')
    for split in member_plan['members']:
        if set(ids) & set(split['train_ids'] + split['validation_ids']):
            raise ValueError('Selection study leakage')
    cfg = manifest['configuration']
    model = cnn.ResNetKnee(pretrained=False, dropout=cfg['dropout']).cuda().eval()
    model.load_state_dict(torch.load(package/'model.pt', map_location='cpu', weights_only=True)['state_dict'], strict=True)
    probabilities = []
    with torch.inference_mode():
        for count, uid in enumerate(ids, 1):
            cnn.check_budget(deadline)
            images = cnn.study_images(uid, by_study, index['identity']['run_id'], cfg)
            with torch.autocast('cuda', dtype=torch.float16):
                z = model(images, cfg['microbatch'])
            probabilities.append(z.float().sigmoid().cpu().numpy()[0])
            if count % 10 == 0: print(f'Member {member}: {count}/100 studies')
    probabilities = np.asarray(probabilities)
    if not np.isfinite(probabilities).all(): raise ValueError('Nonfinite predictions')
    dest = OUTPUT/'regenerated'/f'member-{member}'
    dest.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(probabilities, index=pd.Index(ids,name=data.ID), columns=data.TARGETS).to_csv(dest/'selection_predictions_private.csv')
    targets.to_csv(dest/'selection_targets_private.csv')
    q = np.clip(probabilities,1e-7,1-1e-7)
    bce = float((-(y*np.log(q)+(1-y)*np.log(1-q)))[mask].mean())
    for name, obj in {
        'run_identity.json': identity,
        'split_plan_private.json': member_plan,
        'training_status.json': dict(reason='regenerated_best_checkpoint',best_epoch=manifest['best_epoch']),
        'selection_metrics_private.json': dict(soft_target_bce=bce),
        'model_source.json': dict(package=str(package),manifest_sha256=hashlib.sha256((package/'manifest.json').read_bytes()).hexdigest()),
    }.items():
        (dest/name).write_text(json.dumps(obj,indent=2))
    member_dirs[member] = dest
    print('Member',member,'selection BCE:',bce)
    del model
    torch.cuda.empty_cache()


Member 0 best epoch 15 package /kaggle/input/models/gany24558/gc-rsna-knee-resnet34/pytorch/comm-res-rh25-member-0/2
Member 1 best epoch 10 package /kaggle/input/models/gany24558/gc-rsna-knee-resnet34/pytorch/comm-res-rh25-member-1/2
Member 2 best epoch 12 package /kaggle/input/models/gany24558/gc-rsna-knee-resnet34/pytorch/comm-res-rh25-member-2/2
Member 0: 10/100 studies
Member 0: 20/100 studies
Member 0: 30/100 studies
Member 0: 40/100 studies
Member 0: 50/100 studies
Member 0: 60/100 studies
Member 0: 70/100 studies
Member 0: 80/100 studies
Member 0: 90/100 studies
Member 0: 100/100 studies
Member 0 selection BCE: 0.49062075245476716
Member 1: 10/100 studies
Member 1: 20/100 studies
Member 1: 30/100 studies
Member 1: 40/100 studies
Member 1: 50/100 studies
Member 1: 60/100 studies
Member 1: 70/100 studies
Member 1: 80/100 studies
Member 1: 90/100 studies
Member 1: 100/100 studies
Member 1 selection BCE: 0.4913688577221442
Member 2: 10/100 studies
Member 2: 20/100 studies
Member 2: 

In [4]:
comparison = compare_selection_members(member_dirs, OUTPUT)
display(comparison)
print('Lower BCE is better. Negative delta improves on the best individual member.')
print('Community-label agreement only, not clinical accuracy or an independent test score.')
print('Saved private comparison tables and predictions to', OUTPUT)


,members,model_count,studies,soft_target_bce,bce_delta_vs_best_single
0,0+1+2,3,100,0.486506,-0.004115
1,0+2,2,100,0.486586,-0.004034
2,0+1,2,100,0.487255,-0.003366
3,1+2,2,100,0.488955,-0.001665
4,0,1,100,0.490621,0.000000
5,2,1,100,0.491280,0.000659
6,1,1,100,0.491369,0.000748


Lower BCE is better. Negative delta improves on the best individual member.
Community-label agreement only, not clinical accuracy or an independent test score.
Saved private comparison tables and predictions to /kaggle/working/ensemble_comparison
